# GenerativeAURORA — Exploratory Data Analysis and Data Quality Assessment

## Purpose
Perform exploratory data analysis (EDA) and data-quality checks on the cleaned NSRDB dataset generated by `GenerativeAURORA_merge_nsrdb_Colab.ipynb`.

## Analyses
- Missing values in key meteorological variables
- Sky-regime distribution across study sites
- Average hourly GHI profiles
- Seasonal GHI statistics
- Clearness-index distribution
- Number of observations per site and year

## Workflow position
`download_nsrdb` → `merge_nsrdb` → **`eda_nsrdb`** → sequence construction → model training/evaluation

**Input:** `nsrdb_merged.parquet`

**Outputs:** CSV summary tables in `eda_output/`


## 1. Install and import required packages


In [ ]:
!pip -q install pandas numpy pyarrow

import pandas as pd
import numpy as np
import os

print("Packages imported successfully.")


Packages imported successfully.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Configure input and output paths


In [ ]:
PARQUET_FILE = "/content/drive/MyDrive/Colab Notebooks/merge_nsrdb/nsrdb_merged.parquet"
OUTPUT_DIR = "eda_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Input file :", PARQUET_FILE)
print("Output dir :", OUTPUT_DIR)


Input file : /content/drive/MyDrive/Colab Notebooks/merge_nsrdb/nsrdb_merged.parquet
Output dir : eda_output


## 3. Load the merged NSRDB dataset


In [ ]:
def load_data():
    print("Loading " + PARQUET_FILE + "...")
    if not os.path.exists(PARQUET_FILE):
        raise FileNotFoundError(
            f"{PARQUET_FILE} was not found. Run merge_nsrdb first."
        )

    df = pd.read_parquet(PARQUET_FILE)
    print("Loaded " + str(len(df)) + " rows, " + str(len(df.columns)) + " columns.")
    print("Columns: " + str(list(df.columns)))
    print("")
    return df

df = load_data()


Loading /content/drive/MyDrive/Colab Notebooks/merge_nsrdb/nsrdb_merged.parquet...
Loaded 110502 rows, 22 columns.
Columns: ['YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE', 'GHI', 'DHI', 'DNI', 'WIND SPEED', 'TEMPERATURE', 'RELATIVE HUMIDITY', 'PRESSURE', 'CLOUD TYPE', 'SOLAR ZENITH ANGLE', 'CLEARSKY GHI', 'SITE', 'CLIMATE', 'LAT', 'LON', 'CLEARNESS_INDEX', 'SKY_REGIME', 'DATETIME']



## 4. Missing-value assessment


In [ ]:
def check_missing(df):
    print("=== Missing value check ===")
    key_cols = [
        "GHI", "DHI", "DNI", "WIND SPEED", "AIR TEMPERATURE",
        "RELATIVE HUMIDITY", "CLEARNESS_INDEX", "SKY_REGIME"
    ]

    for col in key_cols:
        if col in df.columns:
            n_missing = df[col].isna().sum()
            pct = round(100.0 * n_missing / len(df), 2)
            status = "OK" if n_missing == 0 else "CHECK"
            print(
                "  " + col.ljust(25)
                + " missing: " + str(n_missing)
                + " (" + str(pct) + "%)  " + status
            )
        else:
            print("  " + col.ljust(25) + " NOT FOUND in dataset")
    print("")

check_missing(df)


=== Missing value check ===
  GHI                       missing: 0 (0.0%)  OK
  DHI                       missing: 0 (0.0%)  OK
  DNI                       missing: 0 (0.0%)  OK
  WIND SPEED                missing: 0 (0.0%)  OK
  AIR TEMPERATURE           NOT FOUND in dataset
  RELATIVE HUMIDITY         missing: 0 (0.0%)  OK
  CLEARNESS_INDEX           missing: 0 (0.0%)  OK
  SKY_REGIME                missing: 0 (0.0%)  OK



## 5. Sky-regime distribution by site


In [ ]:
def regime_distribution(df):
    print("=== Sky regime distribution per site ===")
    site_col = "SITE_NAME" if "SITE_NAME" in df.columns else "SITE"

    if site_col not in df.columns:
        raise ValueError("No SITE_NAME or SITE column found.")
    if "SKY_REGIME" not in df.columns:
        raise ValueError("SKY_REGIME column not found in dataset.")

    table = (
        df.groupby([site_col, "SKY_REGIME"])
        .size()
        .unstack(fill_value=0)
    )

    table["TOTAL"] = table.sum(axis=1)

    for regime in ["Clear", "Cloudy", "Overcast"]:
        if regime in table.columns:
            table[regime + "_PCT"] = (
                100.0 * table[regime] / table["TOTAL"]
            ).round(1)

    print(table.to_string())

    out = os.path.join(OUTPUT_DIR, "regime_distribution.csv")
    table.to_csv(out)
    print("Saved to " + out)
    print("")
    return table

regime_table = regime_distribution(df)


=== Sky regime distribution per site ===
SKY_REGIME     Clear  Cloudy  Overcast  TOTAL  Clear_PCT  Cloudy_PCT  Overcast_PCT
SITE                                                                              
Denver_CO      19826    1948       317  22091       89.7         8.8           1.4
LosAngeles_CA  20631    1343       149  22123       93.3         6.1           0.7
Miami_FL       20216    1498       258  21972       92.0         6.8           1.2
Phoenix_AZ     21403     659       121  22183       96.5         3.0           0.5
Seattle_WA     16772    4856       505  22133       75.8        21.9           2.3
Saved to eda_output/regime_distribution.csv



## 6. Average hourly GHI profile


In [ ]:
def hourly_ghi_profile(df):
    """
    Average GHI by hour-of-day per site and per regime.
    This is the "typical day profile" used in the supplied script.
    """
    print("=== Hourly GHI profile (average per site) ===")
    site_col = "SITE_NAME" if "SITE_NAME" in df.columns else "SITE"
    hour_col = "HOUR" if "HOUR" in df.columns else None

    if hour_col is None:
        print("  WARNING: No HOUR column found. Skipping hourly profile.")
        return None

    profile = (
        df.groupby([site_col, hour_col])["GHI"]
        .mean()
        .round(1)
        .unstack(level=0)
    )

    print(profile.to_string())

    out = os.path.join(OUTPUT_DIR, "hourly_ghi_profile.csv")
    profile.to_csv(out)
    print("Saved to " + out)
    print("")
    return profile

hourly_profile = hourly_ghi_profile(df)


=== Hourly GHI profile (average per site) ===
SITE  Denver_CO  LosAngeles_CA  Miami_FL  Phoenix_AZ  Seattle_WA
HOUR                                                            
4           NaN            NaN       NaN         NaN         8.5
5          59.7           51.8       NaN        11.2        54.3
6         145.0          138.4      66.2        85.8       115.7
7         221.4          243.9     174.2       184.5       183.9
8         374.3          398.9     350.8       343.9       233.9
9         514.2          558.6     513.1       518.0       330.6
10        616.8          677.9     629.6       661.9       414.4
11        666.3          738.8     698.5       755.4       467.5
12        651.9          743.6     712.6       787.1       470.3
13        585.6          683.4     670.6       755.5       442.0
14        472.7          571.6     572.2       663.4       382.1
15        337.8          412.2     434.9       521.1       289.5
16        196.6          231.7     277.3    

## 7. Seasonal GHI statistics

The supplied script uses:
- DJF = December, January, February
- MAM = March, April, May
- JJA = June, July, August
- SON = September, October, November


In [ ]:
def seasonal_stats(df):
    """
    Stats broken down by season (DJF/MAM/JJA/SON).
    Useful to show seasonal distribution shift.
    """
    print("=== Seasonal GHI statistics per site ===")
    site_col = "SITE_NAME" if "SITE_NAME" in df.columns else "SITE"

    if "MONTH" not in df.columns:
        print("  WARNING: No MONTH column. Skipping seasonal stats.")
        return None

    season_map = {
        12: "DJF", 1: "DJF", 2: "DJF",
        3: "MAM", 4: "MAM", 5: "MAM",
        6: "JJA", 7: "JJA", 8: "JJA",
        9: "SON", 10: "SON", 11: "SON",
    }

    seasonal_df = df.copy()
    seasonal_df["SEASON"] = seasonal_df["MONTH"].map(season_map)

    seasonal = (
        seasonal_df.groupby([site_col, "SEASON"])["GHI"]
        .agg(["mean", "std", "max"])
        .round(1)
    )

    print(seasonal.to_string())

    out = os.path.join(OUTPUT_DIR, "seasonal_ghi_stats.csv")
    seasonal.to_csv(out)
    print("Saved to " + out)
    print("")
    return seasonal

seasonal_table = seasonal_stats(df)


=== Seasonal GHI statistics per site ===
                       mean    std   max
SITE          SEASON                    
Denver_CO     DJF     274.6  200.7   803
              JJA     495.1  317.7  1073
              MAM     427.9  304.6  1052
              SON     369.3  253.3   915
LosAngeles_CA DJF     327.9  219.9   834
              JJA     547.9  332.7  1065
              MAM     473.5  322.9  1064
              SON     423.2  264.3   963
Miami_FL      DJF     363.4  237.4   923
              JJA     475.9  301.9  1007
              MAM     484.8  308.6  1049
              SON     392.5  267.1   960
Phoenix_AZ    DJF     343.5  222.7   846
              JJA     539.3  336.3  1087
              MAM     545.6  323.4  1098
              SON     435.4  264.5   944
Seattle_WA    DJF     117.8  115.5   611
              JJA     408.6  293.1   961
              MAM     339.4  264.0   948
              SON     231.4  200.4   795
Saved to eda_output/seasonal_ghi_stats.csv



## 8. Clearness-index distribution

The supplied script uses five descriptive intervals:
- 0.00–0.10 (Overcast)
- 0.10–0.35 (Cloudy)
- 0.35–0.60 (Partly Clear)
- 0.60–0.80 (Mostly Clear)
- 0.80–1.00 (Clear)


In [ ]:
def clearness_index_stats(df):
    """
    Clearness index distribution — key input to sky regime gating.
    This distinguishes the five sites in the supplied analysis.
    """
    print("=== Clearness index distribution per site ===")
    site_col = "SITE_NAME" if "SITE_NAME" in df.columns else "SITE"

    if "CLEARNESS_INDEX" not in df.columns:
        print("  WARNING: CLEARNESS_INDEX column not found.")
        return None

    bins = [0.0, 0.10, 0.35, 0.60, 0.80, 1.01]
    labels = [
        "0.00-0.10 (Overcast)",
        "0.10-0.35 (Cloudy)",
        "0.35-0.60 (Partly Clear)",
        "0.60-0.80 (Mostly Clear)",
        "0.80-1.00 (Clear)"
    ]

    ci_df = df.copy()
    ci_df["CI_BIN"] = pd.cut(
        ci_df["CLEARNESS_INDEX"],
        bins=bins,
        labels=labels,
        right=False
    )

    ci_table = (
        ci_df.groupby([site_col, "CI_BIN"])
        .size()
        .unstack(fill_value=0)
    )

    print(ci_table.to_string())

    out = os.path.join(OUTPUT_DIR, "clearness_index_distribution.csv")
    ci_table.to_csv(out)
    print("Saved to " + out)
    print("")
    return ci_table

ci_table = clearness_index_stats(df)


=== Clearness index distribution per site ===
CI_BIN         0.00-0.10 (Overcast)  0.10-0.35 (Cloudy)  0.35-0.60 (Partly Clear)  0.60-0.80 (Mostly Clear)  0.80-1.00 (Clear)
SITE                                                                                                                          
Denver_CO                       315                1946                      2704                      3186              13940
LosAngeles_CA                   149                1341                      1322                      1553              17758
Miami_FL                        257                1498                      2213                      3244              14760
Phoenix_AZ                      119                 660                      1097                      1863              18444
Seattle_WA                      502                4854                      3798                      3095               9884
Saved to eda_output/clearness_index_distribution.csv



/tmp/ipykernel_1009/1527694335.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ci_df.groupby([site_col, "CI_BIN"])


## 9. Data-quality report: rows per site and year

The supplied script records the number of retained observations per site/year and notes:
- 2020: 8,784 total hours
- Other years: 8,760 total hours

After daytime filtering, the supplied script expects roughly 4,000–5,500 rows per site/year.


In [ ]:
def data_quality_report(df):
    """
    Final data quality summary — rows per site/year, any gaps.
    """
    print("=== Data quality report ===")
    site_col = "SITE_NAME" if "SITE_NAME" in df.columns else "SITE"

    if "YEAR" in df.columns and site_col in df.columns:
        counts = df.groupby([site_col, "YEAR"]).size().unstack(fill_value=0)
        print("Rows per site per year:")
        print(counts.to_string())

        out = os.path.join(OUTPUT_DIR, "rows_per_site_year.csv")
        counts.to_csv(out)
        print("Saved to " + out)

    print("")
    print("Note: 2020 has 8784 total hours (leap year). All other years have 8760.")
    print("After daytime filtering, expect roughly 4000-5500 rows per site per year.")
    print("")

data_quality_report(df)


=== Data quality report ===
Rows per site per year:
YEAR           2018  2019  2020  2021  2022
SITE                                       
Denver_CO      4416  4415  4426  4418  4416
LosAngeles_CA  4423  4422  4433  4423  4422
Miami_FL       4393  4392  4403  4392  4392
Phoenix_AZ     4434  4435  4444  4433  4437
Seattle_WA     4355  4442  4452  4443  4441
Saved to eda_output/rows_per_site_year.csv

Note: 2020 has 8784 total hours (leap year). All other years have 8760.
After daytime filtering, expect roughly 4000-5500 rows per site per year.



## 10. Run the complete EDA workflow

The individual cells above allow each analysis to be inspected separately. The following cell reproduces the complete sequence of the original Python script.


In [ ]:
print("GenerativeAURORA - EDA Script")
print("")

check_missing(df)
regime_distribution(df)
hourly_ghi_profile(df)
seasonal_stats(df)
clearness_index_stats(df)
data_quality_report(df)

print("=== EDA complete ===")
print("All output tables saved to: " + OUTPUT_DIR + "/")


GenerativeAURORA - EDA Script

=== Missing value check ===
  GHI                       missing: 0 (0.0%)  OK
  DHI                       missing: 0 (0.0%)  OK
  DNI                       missing: 0 (0.0%)  OK
  WIND SPEED                missing: 0 (0.0%)  OK
  AIR TEMPERATURE           NOT FOUND in dataset
  RELATIVE HUMIDITY         missing: 0 (0.0%)  OK
  CLEARNESS_INDEX           missing: 0 (0.0%)  OK
  SKY_REGIME                missing: 0 (0.0%)  OK

=== Sky regime distribution per site ===
SKY_REGIME     Clear  Cloudy  Overcast  TOTAL  Clear_PCT  Cloudy_PCT  Overcast_PCT
SITE                                                                              
Denver_CO      19826    1948       317  22091       89.7         8.8           1.4
LosAngeles_CA  20631    1343       149  22123       93.3         6.1           0.7
Miami_FL       20216    1498       258  21972       92.0         6.8           1.2
Phoenix_AZ     21403     659       121  22183       96.5         3.0           0.5
Se

/tmp/ipykernel_1009/1527694335.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ci_df.groupby([site_col, "CI_BIN"])


## 11. Verify generated output files


In [ ]:
expected_outputs = [
    "regime_distribution.csv",
    "hourly_ghi_profile.csv",
    "seasonal_ghi_stats.csv",
    "clearness_index_distribution.csv",
    "rows_per_site_year.csv",
]

print("Files produced:")

for filename in expected_outputs:
    fpath = os.path.join(OUTPUT_DIR, filename)

    if os.path.exists(fpath):
        size_kb = round(os.path.getsize(fpath) / 1024, 1)
        print("  [OK] " + filename.ljust(40) + str(size_kb) + " KB")
    else:
        print("  [MISSING] " + filename)

print("")
print("EDA outputs are ready for review before sequence construction.")
print("Next step: prepare training windows for the diffusion model.")


Files produced:
  [OK] regime_distribution.csv                 0.3 KB
  [OK] hourly_ghi_profile.csv                  0.5 KB
  [OK] seasonal_ghi_stats.csv                  0.6 KB
  [OK] clearness_index_distribution.csv        0.3 KB
  [OK] rows_per_site_year.csv                  0.2 KB

EDA outputs are ready for review before sequence construction.
Next step: prepare training windows for the diffusion model.
